# Hotel Booking Cancellation Prediction - Data Analysis

## Objectives
Analyze hotel booking dataset with focus on:
1. Relationships between features and features
2. Relationships between features and labels (booking cancellation)

## Dataset Description
- Training data: train.csv (17 features + label)
- Test data: test.csv (17 features, no label)
- Label: 0=Not Cancelled, 1=Cancelled


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set font for better display
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Set plot style
sns.set_style("whitegrid")
plt.style.use('seaborn-v0_8')


In [ ]:
# Load data
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
print("\nTraining data columns:")
print(train_data.columns.tolist())


In [ ]:
# Basic data information
print("=== Basic Data Information ===")
print(train_data.info())
print("\n=== Statistical Description ===")
print(train_data.describe())


In [ ]:
# Check missing values
print("=== Missing Values Check ===")
missing_values = train_data.isnull().sum()
print(missing_values[missing_values > 0])

# Label distribution
print("\n=== Label Distribution ===")
label_counts = train_data['label'].value_counts()
print(f"Not Cancelled (0): {label_counts[0]} ({label_counts[0]/len(train_data)*100:.2f}%)")
print(f"Cancelled (1): {label_counts[1]} ({label_counts[1]/len(train_data)*100:.2f}%)")


In [ ]:
# Feature type analysis
print("=== Feature Type Analysis ===")
categorical_features = []
numerical_features = []

for col in train_data.columns:
    if col in ['id', 'label']:
        continue
    if train_data[col].dtype == 'object':
        categorical_features.append(col)
    else:
        numerical_features.append(col)

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")


In [ ]:
# Categorical feature analysis
print("=== Categorical Feature Analysis ===")
for feature in categorical_features:
    print(f"\n{feature}:")
    value_counts = train_data[feature].value_counts()
    print(value_counts.head(10))  # Show top 10 most common values


## 1. Feature-Label Relationship Analysis


In [ ]:
# Numerical features vs label relationship
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.ravel()

for i, feature in enumerate(numerical_features[:9]):
    # Box plot grouped by label
    train_data.boxplot(column=feature, by='label', ax=axes[i])
    axes[i].set_title(f'{feature} vs Cancellation Status')
    axes[i].set_xlabel('Cancellation Status (0=Not Cancelled, 1=Cancelled)')
    axes[i].set_ylabel(feature)

plt.tight_layout()
plt.show()


In [ ]:
# Categorical features vs label relationship
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(categorical_features[:4]):
    # Calculate cancellation rate for each category
    cancellation_rate = train_data.groupby(feature)['label'].mean().sort_values(ascending=False)
    
    cancellation_rate.plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'{feature} Cancellation Rate by Category')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Cancellation Rate')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 2. Feature-Feature Relationship Analysis


In [ ]:
# Numerical features correlation analysis
plt.figure(figsize=(12, 10))
correlation_matrix = train_data[numerical_features].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Numerical Features Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Find highly correlated feature pairs
print("=== Highly Correlated Feature Pairs (|correlation| > 0.5) ===")
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            high_corr_pairs.append((correlation_matrix.columns[i], 
                                 correlation_matrix.columns[j], 
                                 corr_value))

for feature1, feature2, corr in high_corr_pairs:
    print(f"{feature1} - {feature2}: {corr:.3f}")


## 3. In-depth Analysis - Key Features vs Cancellation Rate


In [ ]:
# Lead time vs cancellation rate relationship
plt.figure(figsize=(12, 6))

# Group lead_time
train_data['lead_time_group'] = pd.cut(train_data['lead_time'], 
                                      bins=[0, 7, 30, 90, 365], 
                                      labels=['0-7 days', '7-30 days', '30-90 days', '90+ days'])

cancellation_by_lead_time = train_data.groupby('lead_time_group')['label'].agg(['count', 'mean'])
cancellation_by_lead_time.columns = ['Booking Count', 'Cancellation Rate']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Booking count
cancellation_by_lead_time['Booking Count'].plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Booking Count by Lead Time Period')
ax1.set_xlabel('Lead Time')
ax1.set_ylabel('Booking Count')
ax1.tick_params(axis='x', rotation=45)

# Cancellation rate
cancellation_by_lead_time['Cancellation Rate'].plot(kind='bar', ax=ax2, color='salmon')
ax2.set_title('Cancellation Rate by Lead Time Period')
ax2.set_xlabel('Lead Time')
ax2.set_ylabel('Cancellation Rate')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n=== Lead Time vs Cancellation Rate Analysis ===")
print(cancellation_by_lead_time)


In [ ]:
# Room price vs cancellation rate relationship
plt.figure(figsize=(12, 6))

# Group price
train_data['price_group'] = pd.cut(train_data['avg_price_per_room'], 
                                  bins=[0, 50, 100, 150, 200, 1000], 
                                  labels=['0-50', '50-100', '100-150', '150-200', '200+'])

cancellation_by_price = train_data.groupby('price_group')['label'].agg(['count', 'mean'])
cancellation_by_price.columns = ['Booking Count', 'Cancellation Rate']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Booking count
cancellation_by_price['Booking Count'].plot(kind='bar', ax=ax1, color='lightgreen')
ax1.set_title('Booking Count by Price Range')
ax1.set_xlabel('Price Range')
ax1.set_ylabel('Booking Count')
ax1.tick_params(axis='x', rotation=45)

# Cancellation rate
cancellation_by_price['Cancellation Rate'].plot(kind='bar', ax=ax2, color='orange')
ax2.set_title('Cancellation Rate by Price Range')
ax2.set_xlabel('Price Range')
ax2.set_ylabel('Cancellation Rate')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n=== Price Range vs Cancellation Rate Analysis ===")
print(cancellation_by_price)


In [ ]:
# Special requests vs cancellation rate relationship
plt.figure(figsize=(10, 6))

cancellation_by_special_requests = train_data.groupby('no_of_special_requests')['label'].agg(['count', 'mean'])
cancellation_by_special_requests.columns = ['Booking Count', 'Cancellation Rate']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Booking count
cancellation_by_special_requests['Booking Count'].plot(kind='bar', ax=ax1, color='purple')
ax1.set_title('Booking Count by Special Requests')
ax1.set_xlabel('Number of Special Requests')
ax1.set_ylabel('Booking Count')
ax1.tick_params(axis='x', rotation=0)

# Cancellation rate
cancellation_by_special_requests['Cancellation Rate'].plot(kind='bar', ax=ax2, color='red')
ax2.set_title('Cancellation Rate by Special Requests')
ax2.set_xlabel('Number of Special Requests')
ax2.set_ylabel('Cancellation Rate')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n=== Special Requests vs Cancellation Rate Analysis ===")
print(cancellation_by_special_requests)


## 4. Statistical Analysis Summary


In [ ]:
# Calculate correlation between features and labels
print("=== Feature-Label Correlation Analysis ===")

# Numerical features correlation with label
numerical_corr = train_data[numerical_features + ['label']].corr()['label'].drop('label')
numerical_corr_sorted = numerical_corr.abs().sort_values(ascending=False)

print("\nNumerical features correlation with label (sorted by absolute value):")
for feature, corr in numerical_corr_sorted.items():
    print(f"{feature}: {corr:.4f}")

# Categorical features relationship with label (using chi-square test)
print("\nCategorical features relationship with label (Chi-square test):")
from scipy.stats import chi2_contingency

for feature in categorical_features:
    contingency_table = pd.crosstab(train_data[feature], train_data['label'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    print(f"{feature}: Chi-square={chi2:.4f}, p-value={p_value:.6f}")


In [ ]:
# Generate analysis report
print("\n" + "="*60)
print("                   Data Analysis Summary Report")
print("="*60)

print(f"\n1. Dataset Overview:")
print(f"   - Training samples: {len(train_data):,}")
print(f"   - Test samples: {len(test_data):,}")
print(f"   - Number of features: {len(train_data.columns)-2} (excluding id and label)")
print(f"   - Numerical features: {len(numerical_features)}")
print(f"   - Categorical features: {len(categorical_features)}")

print(f"\n2. Label Distribution:")
print(f"   - Not Cancelled (0): {label_counts[0]:,} ({label_counts[0]/len(train_data)*100:.2f}%)")
print(f"   - Cancelled (1): {label_counts[1]:,} ({label_counts[1]/len(train_data)*100:.2f}%)")

print(f"\n3. Most Correlated Numerical Features with Cancellation (Top 5):")
for i, (feature, corr) in enumerate(numerical_corr_sorted.head(5).items()):
    print(f"   {i+1}. {feature}: {corr:.4f}")

print(f"\n4. Key Findings:")
print(f"   - Lead time shows strong positive correlation with cancellation rate")
print(f"   - Room price relationship with cancellation needs further analysis")
print(f"   - Special requests show strong negative correlation with cancellation")
print(f"   - Customer historical behavior significantly affects current cancellation")

print(f"\n5. Recommendations:")
print(f"   - Focus on customers with long lead times")
print(f"   - Analyze different market segment and room type combinations")
print(f"   - Consider customer historical cancellation behavior as important features")
print(f"   - Customers with many special requests have lower cancellation rates")

print("\n" + "="*60)
